# Hinglish Review Analyzer: transformer fine-tuning (Colab GPU)

Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.

This trains the classical baselines (HMM, CRF, NB, LR, SVM) and fine-tunes multilingual / Hinglish transformers on the **same splits**, then builds `results/comparison.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the project
Either clone your GitHub repo (set `REPO_URL`) or upload `hinglish-absa.zip`.

In [ ]:
REPO_URL = ""  # e.g. https://github.com/<you>/hinglish-absa.git
import os
if REPO_URL:
    !git clone -q $REPO_URL hinglish-absa
else:
    from google.colab import files
    up = files.upload()
    zipname = next(iter(up))
    !unzip -q -o "$zipname"
%cd /content/hinglish-absa

In [ ]:
!pip install -q -r requirements.txt -r requirements-transformers.txt
!bash scripts/download_data.sh

## 2. Classical baselines (CPU part, ~5 min)

In [ ]:
!python scripts/train_all.py

## 3. Fine-tune transformers
`MODELS` and `TASKS` can be trimmed if you are short on time. Rough T4 time: LID ~6 min/model, NER ~2 min, ACD/ACSC ~3 min each.

In [ ]:
MODELS = [
    "bert-base-multilingual-cased",   # mBERT
    "l3cube-pune/hing-bert",          # HingBERT (Roman Hinglish)
    "l3cube-pune/hing-mbert-mixed",   # HingMBERT (Roman + Devanagari)
    "google/muril-base-cased",        # MuRIL
]
TASKS = ["lid", "ner", "acd", "acsc"]
EPOCHS = {"lid": 2, "ner": 5, "acd": 5, "acsc": 4}

for task in TASKS:
    for m in MODELS:
        print(f"===== {task} | {m}")
        !python scripts/finetune_transformer.py --task $task --model $m --epochs {EPOCHS[task]} --batch-size 32 --fp16 2>&1 | grep -E "epoch|TEST|Error"

## 4. Comparison table

In [ ]:
!python scripts/compare_results.py

## 5. (Optional) save the best ACSC model and download results

In [ ]:
# !python scripts/finetune_transformer.py --task acsc --model l3cube-pune/hing-mbert-mixed --epochs 4 --fp16 --save-model
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")